# Loss Given Default (LGD) Regression

### Credit Scoring Risk Assessment — Banking-AI

This notebook explains each step in plain language so new learners can follow:

## Step 0 — Notebook Header

**Prerequisites:** Basic Python (`pandas`, `numpy`, `matplotlib`), familiarity with `scikit-learn` basics, and basic understanding of credit scoring and risk assessment terminology.

**What You Will Learn:**

After completing this notebook, you will be able to:
- Explain the credit scoring and risk assessment prediction task and why regression modelling supports this banking decision.
- Implement a regression workflow and evaluate predictive accuracy using domain-appropriate error metrics.
- Evaluate whether the prediction error is acceptable for the operational decision it supports.

**Estimated time:** 35–45 minutes

**Collection context:** Credit scoring, probability of default, loss estimation, and stress testing for banking risk management.

## Step 1 — Banking Problem Introduction

**Why this notebook matters:** If a borrower defaults on a \$100,000 loan, the bank doesn't necessarily lose all \$100,000. If the borrower had a car worth \$30,000 as collateral, the bank sells it and "recovers" some money. LGD is the % that is *not* recovered. Predicting LGD helps banks value their collateral and set loan terms.

**Input data used:** Collateral type (Real Estate, Vehicle, Unsecured), Loan-to-Value (LTV) at default, recovery time (months), and legal costs.

**What we predict:** Continuous LGD percentage (0.0 to 1.0).

**ML method used:** Random Forest Regressor.

**Learning goal:** Learn how to model "Recovery Risk" using regression.

**Primary stakeholders:** credit analysts, loan officers, risk managers, and regulatory auditors.

**Comprehension check:** Before looking at the data, write one sentence describing the core decision this model supports and who benefits from getting it right.

**Domain framing note:** This notebook uses synthetic data for educational purposes. It demonstrates decision-support prototyping, not production-ready deployment.

## Step 2 — Environment Setup

Import libraries and set deterministic seeds for reproducibility.

In [ ]:
# Requires: numpy>=1.24, pandas>=2.0, scikit-learn>=1.3, matplotlib>=3.7, seaborn>=0.13

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.preprocessing import LabelEncoder

sns.set_theme(style="whitegrid")

RANDOM_SEED = 42
RNG = np.random.default_rng(RANDOM_SEED)
plt.rcParams["figure.figsize"] = (10, 6)

print("Environment ready.")


## Step 3 — Data Creation & Context

Build Default & Recovery Dataset

We simulate 2,000 defaulted loans. Unsecured loans (no collateral) have very high LGD.

In [ ]:
n = 2000
rng = RNG

collateral_type = rng.choice(['Real Estate', 'Vehicle', 'Unsecured'], n, p=[0.4, 0.4, 0.2])
ltv_at_default = rng.uniform(0.5, 1.5, n)
recovery_time_months = rng.integers(6, 36, n)
legal_costs_pct = rng.uniform(0.02, 0.10, n)

# Logic for LGD (0.0 = full recovery, 1.0 = total loss)
base_lgd = np.where(collateral_type == 'Real Estate', 0.15,
           np.where(collateral_type == 'Vehicle', 0.45, 0.90))

lgd = (
    base_lgd + 
    0.2 * (ltv_at_default - 0.8) + 
    0.005 * recovery_time_months + 
    1.0 * legal_costs_pct + 
    rng.normal(0, 0.05, n)
).clip(0, 1)

df = pd.DataFrame({
    'collateral_type': collateral_type,
    'ltv_at_default': ltv_at_default,
    'recovery_time_months': recovery_time_months,
    'legal_costs_pct': legal_costs_pct,
    'lgd': lgd
})

print("Average LGD by Collateral Type:")
print(df.groupby('collateral_type')['lgd'].mean().round(3))

## Step 4 — Exploratory Data Analysis

EDA

Notice the clear separation in LGD based on whether the loan had a house/car backing it.

In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(x='collateral_type', y='lgd', data=df)
plt.title('Loss Given Default (LGD) by Collateral Type')
plt.ylabel('LGD (Percentage of Exposure Lost)')
plt.tight_layout()
plt.show()
plt.close()

**Alt-text:** Distribution plot titled "Loss Given Default (LGD) by Collateral Type". The chart highlights distributional differences between groups that inform the modelling approach.

## Step 5 — Feature Engineering

Prepare features for modelling. Domain-meaningful transformations help the model learn patterns aligned with banking practice.

In [ ]:
df_encoded = df.copy()
le = LabelEncoder()
df_encoded['collateral_type'] = le.fit_transform(df['collateral_type'])

X = df_encoded.drop('lgd', axis=1)
y = df_encoded['lgd']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_SEED)

## Step 6 — Baseline Establishment

A baseline sets the floor that any useful model must beat. Without it, performance numbers have no context.

In [ ]:
# --- Baseline: always predict the training-set mean ---
baseline_reg = DummyRegressor(strategy='mean')
baseline_reg.fit(X_train, y_train)
baseline_pred_bl = baseline_reg.predict(X_test)
baseline_rmse = np.sqrt(mean_squared_error(y_test, baseline_pred_bl))
print(f"Baseline RMSE (predict-mean): {baseline_rmse:.4f}")
print("Any useful model must produce a lower RMSE.")

## Step 7 — Model Training & Evaluation

Train the model and compare performance against the baseline.

**Prediction prompt:** Before viewing the results, what performance do you expect and why?

In [ ]:
model = RandomForestRegressor(n_estimators=100, random_state=RANDOM_SEED)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

In [ ]:
print(f"Mean Absolute Error: {mean_absolute_error(y_test, y_pred):.4f}")
print(f"R-Squared Score: {r2_score(y_test, y_pred):.4f}")

plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred, alpha=0.4)
plt.plot([0, 1], [0, 1], '--r')
plt.xlabel('Actual LGD')
plt.ylabel('Predicted LGD')
plt.title('LGD Prediction Performance')
plt.tight_layout()
plt.show()
plt.close()

**Alt-text:** Scatter plot, line chart titled "LGD Prediction Performance" with Actual LGD on the x-axis and Predicted LGD on the y-axis. The chart highlights spatial separation or clustering patterns in the data.

## Step 8 — Interpretability & Explainability

Interpret the model in domain terms. A banking professional should be able to assess whether the model's reasoning is credible.

**Summary:**
- **Collateral is King:** Unsecured loans have the highest LGD, meaning the bank loses almost everything if the borrower stops paying.
- **Recovery Time:** Longer recovery times increase LGD due to the time-value of money and continued legal costs.

**Banking Context:** Banks use these models to decide the "Haircut" they apply to collateral. If a car is worth \$10k but the predicted LGD is 50%, the bank might only count it as \$5k of protection.

## Step 9 — Operational Application

Demonstrate how the model output feeds into a real banking workflow decision.

In [ ]:
# --- Scenario: inspect predictions at different points ---
print("Operational demonstration — model predictions across the test range:\n")
low_idx  = int(np.argmin(y_test.values))
high_idx = int(np.argmax(y_test.values))
mid_idx  = int(np.argsort(y_test.values)[len(y_test) // 2])

for label, idx in [("Low-end", low_idx), ("Mid-range", mid_idx), ("High-end", high_idx)]:
    actual = y_test.values[idx]
    pred   = y_pred[idx]
    print(f"  {label}  actual: {actual:.4f}  predicted: {pred:.4f}  error: {abs(actual-pred):.4f}")

print("\nPractitioners would use these predictions as one input among several.")

## Step 10 — Summary & Reflection

**What you accomplished:** You built an end-to-end credit scoring and risk assessment workflow — from problem framing through data creation, baseline comparison, model training, evaluation, and operational demonstration.

**Limitations:**

- The dataset is synthetic and cannot capture the full complexity of real banking data.
- Model performance on synthetic data does not guarantee equivalent performance in production.
- This notebook is an educational prototype. Real deployment requires validated data, regulatory review, and ongoing monitoring.

**Ethical reflection:**

- Credit models can perpetuate historical discrimination if trained on biased lending data.
- Proxy variables such as zip code or employment type may correlate with protected characteristics.
- Applicants deserve transparent explanations of adverse credit decisions under fair lending regulations.

**Reflection questions:**

1. What additional data sources or features would you need before trusting this model in a live banking environment?
2. How would the relative cost of false positives versus false negatives change the metric you optimise for?
3. How could you adapt this workflow to a related problem in credit scoring and risk assessment?

## References

- Scikit-learn documentation: [https://scikit-learn.org/stable/](https://scikit-learn.org/stable/)
- Project quality baseline: `QUALITY_STANDARD.md`
- Domain context drawn from public banking and financial services literature.